# Experimento 12.01 — Viabilidade de treinamento em 2020

**Propósito:** diagnosticar, sem treinar e sem alterar as bases originais, se a coleta de 29/10/2020 permite montar um conjunto de treinamento temporal confiável.

**Atenção:** não classificar todos os endereços diferentes de Tornado como negativos. Esses endereços têm rótulo desconhecido, exceto quando auditados. Apenas uma data não oferece várias janelas diárias para cada pool.


## 1. Ambiente e escolha segura do projeto

In [2]:
from pathlib import Path
import sys, importlib.util, subprocess, hashlib, json, datetime
if importlib.util.find_spec('pyarrow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
BASE_MANUAL = None
ESCOLHA_INDICE = None
ROOT=Path('/content/drive/MyDrive')
candidatos=sorted(p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith('mixers-ethereum')
                and (p/'data/processed/features_teste_20201029.parquet').exists())
print('Cópias com dataset 2020:')
for i,p in enumerate(candidatos): print(i,str(p))
if BASE_MANUAL:
    BASE=Path(BASE_MANUAL)
elif len(candidatos)==1:
    BASE=candidatos[0]
elif len(candidatos)>1:
    if ESCOLHA_INDICE is None:
        raise RuntimeError('Há várias cópias: configure ESCOLHA_INDICE explicitamente.')
    BASE=candidatos[ESCOLHA_INDICE]
else:
    raise FileNotFoundError('Adicione um atalho da pasta ao Meu Drive ou configure BASE_MANUAL')
FEAT=BASE/'data/processed/features_teste_20201029.parquet'
TX=BASE/'data/interim/test_20201029.parquet'
assert FEAT.is_file() and TX.is_file()
print('Base selecionada:',BASE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cópias com dataset 2020:
0 /content/drive/MyDrive/mixers-ethereum
1 /content/drive/MyDrive/mixers-ethereum (1)


RuntimeError: Há várias cópias: configure ESCOLHA_INDICE explicitamente.

## 2. Checagem dos arquivos e das quatro features

In [ ]:
FEATURES=['value_entropy','fee_entropy','internal_value_entropy','call_count_mean']
POOLS={
 '0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc':'Tornado 0,1 ETH',
 '0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936':'Tornado 1 ETH',
 '0x910cbd523d972eb0a6f4cae4618ad62622b39dbf':'Tornado 10 ETH',
 '0xa160cdab225685da1d56aa342ad8841c3b53f291':'Tornado 100 ETH',
}
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(2**20), b''):
            h.update(chunk)
    return h.hexdigest()
print('features sha256:',sha(FEAT))
print('transações sha256:',sha(TX))
df=pd.read_parquet(FEAT)
df['recipient']=df['recipient'].astype(str).str.lower()
assert set(FEATURES+['recipient','dia','n_tx']).issubset(df.columns)
if df['recipient'].duplicated().any():
    print('AVISO: endereço aparece em múltiplas linhas; conferir granularidade por dia.')
print('Janelas totais:',len(df))
print('Janelas n_tx >= 10:',int((df.n_tx>=10).sum()))
print('Nulos:',df[FEATURES].isna().sum().to_dict())
print('Intervalo de dias:',df.dia.min(),df.dia.max())


## 3. Pools conhecidos e elegibilidade

In [ ]:
df['pool_referencia']=df.recipient.map(POOLS)
pool_rows=[]
for endereco,nome in POOLS.items():
    sub=df[df.recipient==endereco]
    pool_rows.append({'pool':nome,'endereco':endereco,'janelas_existentes':len(sub),
                      'janelas_elegiveis_10':int((sub.n_tx>=10).sum()),
                      'total_tx_observadas':int(sub.n_tx.sum()),
                      'max_tx_janela':int(sub.n_tx.max()) if len(sub) else 0})
pools=pd.DataFrame(pool_rows)
display(pools)
display(df.loc[df.pool_referencia.notna(),['recipient','dia','n_tx','pool_referencia']+FEATURES])


## 4. Características: pools vs universo não rotulado

In [ ]:
ativos=df[df.n_tx>=10].copy()
p=ativos[ativos.pool_referencia.notna()]
u=ativos[ativos.pool_referencia.isna()]
print('Pools referência elegíveis:',len(p))
print('Universo adicional NÃO ROTULADO:',len(u))
if len(p):
    res=pd.DataFrame({'pools_mediana':p[FEATURES].median(),
                      'nao_rotulados_mediana':u[FEATURES].median(),
                      'pools_min':p[FEATURES].min(),
                      'pools_max':p[FEATURES].max()})
    display(res)
else:
    res=pd.DataFrame()
    print('Nenhum pool de referência elegível nesta coleta; precisaríamos de outros dias.')


## 5. Conferir características brutas somente dos quatro pools

In [ ]:
tx=pq.read_table(TX,columns=['hash','time','recipient','sender','call_count','value_eth','internal_value_eth','fee_eth']).to_pandas()
tx['recipient']=tx['recipient'].astype(str).str.lower()
txp=tx[tx.recipient.isin(POOLS)].copy()
print('Transações diretamente destinadas aos quatro pools:',len(txp))
print('Contagem por pool:')
display(txp.groupby('recipient').size().rename('n_tx_diretas').reset_index())
print('Nulos nas transações dos pools:')
display(txp[['call_count','value_eth','internal_value_eth','fee_eth']].isna().sum().to_frame('n_nulos'))
print('IMPORTANTE: transações de topo via Router podem não estar incluídas nesta atribuição direta.')
del tx


## 6. Decisão de viabilidade

In [ ]:
n_pools=int((pools.janelas_elegiveis_10>0).sum())
n_pos=int(pools.janelas_elegiveis_10.sum())
print('Pools de referência elegíveis:',n_pools)
print('Janelas positivas disponíveis nesta data:',n_pos)
if n_pos<10 or n_pools<3:
    print('NÃO RECOMENDADO treinar classificador definitivo apenas com esta data.')
else:
    print('Há positivos, mas continuam insuficientes para concluir independência temporal e por contrato.')
print('PRÓXIMA COLETA: múltiplos dias de 2020 com o mesmo processo de atribuição direta e via Router.')


## 7. Salvar somente o diagnóstico em pasta nova

In [ ]:
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out=BASE/'temporal_2020_2026'/'results'/('viabilidade_2020_'+stamp)
out.mkdir(parents=True,exist_ok=False)
pools.to_csv(out/'pools_2020.csv',index=False)
df.loc[df.pool_referencia.notna(),['recipient','dia','n_tx','pool_referencia']+FEATURES].to_csv(out/'features_pools_2020.csv',index=False)
if len(res): res.to_csv(out/'comparacao_features_2020.csv')
manifest={'treino_realizado':False,'arquivo_features':str(FEAT),'sha256_features':sha(FEAT),
          'arquivo_transacoes':str(TX),'sha256_transacoes':sha(TX),
          'n_janelas':len(df),'n_janelas_elegiveis':int((df.n_tx>=10).sum()),
          'n_pools_ativos_elegiveis':n_pools,'n_janelas_positivas_referencia':n_pos,
          'observacao':'Análise direta por recipient; não afirmar negativos nem cobertura via Router'}
(out/'manifesto.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Diagnóstico salvo:',out)
